# CAWT v2 Best - Single Training (Free Colab T4)

**Same structure as cawt_final, but upgraded:**
- SMOTE-Tomek instead of plain SMOTE (removes noisy samples)
- No WeightedRandomSampler (avoids double-balancing)
- Focal Loss + Label Smoothing
- Signal augmentations + Mixup on GPU
- CosineAnnealingWarmRestarts LR scheduler
- AMP (fp16) via GradScaler
- torch.compile() for T4 speed
- persistent_workers + pin_memory
- Test-Time Augmentation x10 at inference
- TorchScript export for Android

In [ ]:
# Cell 1: Install
!pip install -q wfdb imbalanced-learn scikit-learn seaborn matplotlib

In [ ]:
# Cell 2: Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: GPU Check + Imports
import torch, time, math, copy, gc, random, os, glob
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast, GradScaler
from scipy import signal as scipy_signal
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report)
from imblearn.combine import SMOTETomek
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb, warnings
warnings.filterwarnings('ignore', category=UserWarning)

if not torch.cuda.is_available():
    raise RuntimeError(
        'NO GPU! Go to Runtime > Change runtime type > GPU (T4) then restart.')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = 'cuda'

print('=' * 55)
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Torch: {torch.__version__}')
_ = torch.randn(1000, 1000, device='cuda') @ torch.randn(1000, 1000, device='cuda')
torch.cuda.synchronize()
print('GPU warm-up: OK')
print('=' * 55)

In [ ]:
# Cell 4: Config
CONFIG = {
    'path_mitbih'    : '/content/drive/MyDrive/mit-bih-arrhythmia-database-1.0.0',
    'input_length'   : 187,
    'sampling_rate'  : 360,
    'num_classes'    : 5,
    # Architecture
    'in_channels'    : 1,
    'd_model'        : 128,
    'num_heads'      : 4,
    'num_layers'     : 3,
    'drop_path_rate' : 0.15,
    'dropout'        : 0.25,
    # Training (tuned for FREE Colab T4)
    'batch_size'     : 256,
    'epochs'         : 100,
    'max_lr'         : 3e-4,
    'min_lr'         : 1e-6,
    'T_0'            : 25,
    'weight_decay'   : 0.05,
    'label_smoothing': 0.1,
    'focal_gamma'    : 2.0,
    'patience'       : 15,
    'mixup_alpha'    : 0.3,
    'aug_prob'       : 0.5,
    'tta_n'          : 10,
    'num_workers'    : 2,
}

AAMI = {'N':0,'L':0,'R':0,'e':0,'j':0,
        'A':1,'a':1,'J':1,'S':1,
        'V':2,'E':2,
        'F':3,
        '/':4,'f':4,'Q':4}
CLASS_NAMES = ['Normal (N)', 'Supraventricular (S)', 'Ventricular (V)',
               'Fusion (F)', 'Unknown (Q)']
print(f'Config: batch={CONFIG["batch_size"]} epochs={CONFIG["epochs"]}')

In [ ]:
# Cell 5: Beat Extraction
def extract_beats(base_path, sr=360, L=187):
    dat_files = glob.glob(os.path.join(base_path, '*.dat'))
    recs = sorted(set([os.path.splitext(f)[0] for f in dat_files]))
    print(f'Found {len(recs)} records in {base_path}')
    sb, sa = int(0.25 * sr), int(0.27 * sr)
    beats, labels = [], []
    for rp in recs:
        try:
            rec = wfdb.rdrecord(rp)
            ann = wfdb.rdann(rp, 'atr')
            li = rec.sig_name.index('II') if 'II' in rec.sig_name else 0
            sig = rec.p_signal[:, li]
            for i, peak in enumerate(ann.sample):
                sym = ann.symbol[i]
                if sym not in AAMI:
                    continue
                l, r = peak - sb, peak + sa
                if l < 0 or r > len(sig):
                    continue
                beat = sig[l:r]
                if len(beat) != L:
                    beat = scipy_signal.resample(beat, L)
                rng = beat.max() - beat.min()
                if rng > 1e-8:
                    beat = (beat - beat.min()) / rng
                else:
                    beat = np.zeros(L)
                beats.append(beat.astype(np.float32))
                labels.append(AAMI[sym])
        except Exception as e:
            print(f'  skip {rp}: {e}')
    X = np.array(beats, dtype=np.float32)
    y = np.array(labels, dtype=np.int64)
    c = np.bincount(y, minlength=5)
    print(f'Total: {len(X)} beats | N={c[0]} S={c[1]} V={c[2]} F={c[3]} Q={c[4]}')
    return X, y

X_all, y_all = extract_beats(CONFIG['path_mitbih'])

In [ ]:
# Cell 6: Train/Val Split + SMOTE-Tomek on Train ONLY
# Same as cawt_final but SMOTE-Tomek instead of plain SMOTE
# and NO WeightedRandomSampler (avoids double-balancing)

# 1. Split: 80% train, 20% val (same as original)
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=SEED)

print('Before SMOTE-Tomek:')
c_tr = np.bincount(y_train, minlength=5)
c_vl = np.bincount(y_val, minlength=5)
print(f'  Train: {len(X_train)} | N={c_tr[0]} S={c_tr[1]} V={c_tr[2]} F={c_tr[3]} Q={c_tr[4]}')
print(f'  Val  : {len(X_val)} | N={c_vl[0]} S={c_vl[1]} V={c_vl[2]} F={c_vl[3]} Q={c_vl[4]}')

# 2. SMOTE-Tomek on training set ONLY (never touch validation!)
print('\nApplying SMOTE-Tomek to training set (~5-10 min)...')
t0 = time.time()
smt = SMOTETomek(random_state=SEED)
X_train_flat = X_train.reshape(len(X_train), -1)
X_train_res, y_train_res = smt.fit_resample(X_train_flat, y_train)
X_train = X_train_res.reshape(-1, CONFIG['input_length']).astype(np.float32)
y_train = y_train_res.astype(np.int64)

c_tr = np.bincount(y_train, minlength=5)
print(f'Done in {time.time()-t0:.1f}s')
print(f'After SMOTE-Tomek:')
print(f'  Train: {len(X_train)} | N={c_tr[0]} S={c_tr[1]} V={c_tr[2]} F={c_tr[3]} Q={c_tr[4]}')
print(f'  ALL CLASSES BALANCED!')

# 3. Class weights for Focal Loss
cw = 1.0 / (c_tr.astype(np.float32) + 1e-6)
cw = cw / cw.sum() * 5
CLASS_WEIGHTS = torch.tensor(cw, dtype=torch.float32)
print(f'\nFocal Loss weights: {cw.round(3)}')

In [ ]:
# Cell 7: Dataset + DataLoader (NO WeightedRandomSampler)

class ECGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i].unsqueeze(0), self.y[i]

train_ds = ECGDataset(X_train, y_train)
val_ds = ECGDataset(X_val, y_val)

# NO WeightedRandomSampler - SMOTE-Tomek already balanced the data
train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'],
                          shuffle=True, num_workers=CONFIG['num_workers'],
                          pin_memory=True, persistent_workers=True,
                          prefetch_factor=2)
val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'],
                        shuffle=False, num_workers=CONFIG['num_workers'],
                        pin_memory=True, persistent_workers=True,
                        prefetch_factor=2)

print(f'Train: {len(train_loader)} batches | Val: {len(val_loader)} batches')

In [ ]:
# Cell 8: CAWT Architecture (Bidirectional Cross-Attention)

class RoPE1D(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.register_buffer('inv_freq', 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim)))
    def forward(self, x):
        t = torch.arange(x.size(1), device=x.device).type_as(self.inv_freq)
        freqs = torch.einsum('i,j->ij', t, self.inv_freq)
        return torch.cat((freqs, freqs), dim=-1)

def apply_rotary(q, k, freqs):
    f = freqs.unsqueeze(0).unsqueeze(0)
    cos, sin = f.cos(), f.sin()
    def rot(x):
        x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        return torch.cat((-x2, x1), -1)
    return q * cos + rot(q) * sin, k * cos + rot(k) * sin

class DropPath(nn.Module):
    def __init__(self, p=0.):
        super().__init__()
        self.p = p
    def forward(self, x):
        if self.p == 0 or not self.training:
            return x
        keep = 1 - self.p
        return x * x.new_empty((x.shape[0],) + (1,) * (x.ndim - 1)).bernoulli_(keep) / keep

class WaveletExtractor(nn.Module):
    def __init__(self, in_ch, d):
        super().__init__()
        sub = d // 4
        self.stem = nn.Conv1d(in_ch, d // 2, 7, stride=2, padding=3)
        self.b1 = nn.ModuleList([nn.Conv1d(d // 2, sub, k, padding=k // 2) for k in [3, 7, 15, 31]])
        self.bn1 = nn.BatchNorm1d(d)
        self.pool = nn.MaxPool1d(2)
        self.b2 = nn.ModuleList([nn.Conv1d(d, sub, k, padding=k // 2) for k in [3, 7, 15, 31]])
        self.bn2 = nn.BatchNorm1d(d)
    def forward(self, x):
        x = F.gelu(self.stem(x))
        x = F.gelu(self.bn1(torch.cat([b(x) for b in self.b1], 1)))
        x = self.pool(x)
        x = F.gelu(self.bn2(torch.cat([b(x) for b in self.b2], 1)))
        return x.permute(0, 2, 1)

class TimeExtractor(nn.Module):
    def __init__(self, in_ch, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, d // 2, 7, stride=2, padding=3), nn.BatchNorm1d(d // 2), nn.GELU(),
            nn.Conv1d(d // 2, d, 5, padding=2), nn.BatchNorm1d(d), nn.GELU(),
            nn.MaxPool1d(2),
            nn.Conv1d(d, d, 3, padding=1), nn.BatchNorm1d(d), nn.GELU())
    def forward(self, x):
        return self.net(x).permute(0, 2, 1)

class CrossAttn(nn.Module):
    def __init__(self, d, h, drop=0.2):
        super().__init__()
        self.h = h; self.hd = d // h
        self.qkv = nn.ModuleList([nn.Linear(d, d) for _ in range(3)])
        self.out = nn.Linear(d, d)
        self.drop = nn.Dropout(drop)
    def forward(self, q, k, v, freqs=None):
        B, Lq, _ = q.shape; _, Lk, _ = k.shape
        Q = self.qkv[0](q).view(B, Lq, self.h, self.hd).transpose(1, 2)
        K = self.qkv[1](k).view(B, Lk, self.h, self.hd).transpose(1, 2)
        V = self.qkv[2](v).view(B, Lk, self.h, self.hd).transpose(1, 2)
        if freqs is not None:
            Q, K = apply_rotary(Q, K, freqs)
        a = self.drop((Q @ K.transpose(-2, -1) / math.sqrt(self.hd)).softmax(-1))
        return self.out((a @ V).transpose(1, 2).contiguous().view(B, Lq, -1))

class CrossBlock(nn.Module):
    def __init__(self, d, h, dp=0.1, drop=0.2):
        super().__init__()
        self.n1q = nn.LayerNorm(d); self.n1kv = nn.LayerNorm(d)
        self.attn = CrossAttn(d, h, drop); self.dp = DropPath(dp)
        self.n2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(
            nn.Linear(d, d * 4), nn.GELU(), nn.Dropout(drop),
            nn.Linear(d * 4, d), nn.Dropout(drop))
    def forward(self, q, kv, freqs=None):
        q = q + self.dp(self.attn(self.n1q(q), self.n1kv(kv), self.n1kv(kv), freqs))
        return q + self.dp(self.mlp(self.n2(q)))

class CAWT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d = cfg['d_model']; h = cfg['num_heads']; n = cfg['num_layers']
        dp = cfg['drop_path_rate']; drop = cfg['dropout']
        self.wavelet = WaveletExtractor(cfg['in_channels'], d)
        self.time = TimeExtractor(cfg['in_channels'], d)
        self.rope = RoPE1D(d // h)
        dpr = [dp * i / max(n - 1, 1) for i in range(n)]
        self.w2t = nn.ModuleList([CrossBlock(d, h, dpr[i], drop) for i in range(n)])
        self.t2w = nn.ModuleList([CrossBlock(d, h, dpr[i], drop) for i in range(n)])
        self.norm = nn.LayerNorm(d * 2)
        self.head = nn.Sequential(
            nn.Linear(d * 2, d), nn.GELU(), nn.Dropout(drop),
            nn.Linear(d, cfg['num_classes']))
    def forward(self, x):
        w = self.wavelet(x)
        t = self.time(x)
        freqs = self.rope(w)
        for w2t_blk, t2w_blk in zip(self.w2t, self.t2w):
            w_new = w2t_blk(w, t, freqs)  # wavelet attends to time
            t_new = t2w_blk(t, w, freqs)  # time attends to wavelet
            w, t = w_new, t_new
        return self.head(self.norm(torch.cat([w.mean(1), t.mean(1)], 1)))

with torch.no_grad():
    _m = CAWT(CONFIG); _o = _m(torch.randn(2, 1, 187))
    assert _o.shape == (2, 5)
    params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
    print(f'CAWT OK | {params:,} params')
del _m, _o

In [ ]:
# Cell 9: Focal Loss + Augmentor + Mixup + Metrics

class FocalLoss(nn.Module):
    def __init__(self, nc, alpha, gamma=2.0, smoothing=0.1):
        super().__init__()
        self.nc = nc; self.gamma = gamma; self.s = smoothing
        self.register_buffer('alpha', alpha)
    def forward(self, logits, targets):
        with torch.no_grad():
            sm = torch.full_like(logits, self.s / (self.nc - 1))
            sm.scatter_(1, targets.unsqueeze(1), 1.0 - self.s)
        lp = F.log_softmax(logits, -1)
        focal = (1 - lp.exp()).pow(self.gamma)
        return ((-focal * lp * sm).sum(-1) * self.alpha[targets]).mean()

class Augmentor:
    def __init__(self, p=0.5):
        self.p = p
    def __call__(self, x):
        if torch.rand(1) < self.p:
            x = x + torch.randn_like(x) * 0.02
        if torch.rand(1) < self.p:
            L = x.shape[-1]
            t = torch.linspace(0, 2 * math.pi, L, device=x.device)
            x = x + (torch.sin(t * torch.rand(1, device=x.device) * 3) * 0.05).unsqueeze(0)
        if torch.rand(1) < self.p:
            x = x * (0.85 + torch.rand(1, device=x.device) * 0.30)
        if torch.rand(1) < self.p:
            L = x.shape[-1]; cut = int(L * 0.12)
            s = torch.randint(0, L - cut, (1,)).item()
            x[..., s:s+cut] = 0
        if torch.rand(1) < self.p:
            x = torch.roll(x, torch.randint(-8, 9, (1,)).item(), dims=-1)
        return x

def do_mixup(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def calc_metrics(yt, yp, ypr, nc=5):
    acc = accuracy_score(yt, yp)
    f1 = f1_score(yt, yp, average='macro', zero_division=0)
    try: auc = roc_auc_score(yt, ypr, multi_class='ovr')
    except: auc = 0.0
    cm = confusion_matrix(yt, yp, labels=list(range(nc)))
    sens, spec = [], []
    for i in range(nc):
        tp = cm[i, i]; fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp; tn = cm.sum() - tp - fp - fn
        sens.append(tp / (tp + fn + 1e-9))
        spec.append(tn / (tn + fp + 1e-9))
    return acc, f1, auc, sens, spec

print('Focal Loss, Augmentor, Mixup, Metrics ready.')

In [ ]:
# Cell 10: Training Loop (single run, like cawt_final)

model = CAWT(CONFIG).to(DEVICE)

# torch.compile() for T4 speed boost
if hasattr(torch, 'compile'):
    try:
        model = torch.compile(model, mode='reduce-overhead')
        print('torch.compile() enabled')
    except Exception as e:
        print(f'torch.compile skipped: {e}')

criterion = FocalLoss(CONFIG['num_classes'], CLASS_WEIGHTS.to(DEVICE),
                      CONFIG['focal_gamma'], CONFIG['label_smoothing'])
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['max_lr'],
                        weight_decay=CONFIG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=CONFIG['T_0'], T_mult=1, eta_min=CONFIG['min_lr'])
scaler = GradScaler('cuda')
aug = Augmentor(CONFIG['aug_prob'])

best_f1 = 0.0; best_state = None; patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': [], 'val_auc': [], 'lr': []}

print(f'\nStarting training: {CONFIG["epochs"]} epochs on {DEVICE}')
print(f'Train: {len(train_loader.dataset)} samples | Val: {len(val_loader.dataset)} samples')
print('=' * 65)

for epoch in range(CONFIG['epochs']):
    t0 = time.time()

    # --- Train ---
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        x = aug(x)
        use_mix = np.random.rand() < 0.4
        if use_mix:
            x, ya, yb, lam = do_mixup(x, y, CONFIG['mixup_alpha'])
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            logits = model(x)
            if use_mix:
                loss = lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb)
            else:
                loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item() * x.size(0)
    scheduler.step()
    train_loss /= len(train_loader.dataset)

    # --- Validate ---
    model.eval()
    val_loss = 0.0
    preds_list, trues_list, probs_list = [], [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE, non_blocking=True)
            y_dev = y.to(DEVICE, non_blocking=True)
            with autocast('cuda'):
                logits = model(x)
                v_loss = criterion(logits, y_dev)
            val_loss += v_loss.item() * x.size(0)
            p = logits.float().softmax(1)
            probs_list.extend(p.cpu().numpy())
            preds_list.extend(p.argmax(1).cpu().numpy())
            trues_list.extend(y.numpy())
    val_loss /= len(val_loader.dataset)

    acc, f1, auc, sens, spec = calc_metrics(
        np.array(trues_list), np.array(preds_list), np.array(probs_list))

    lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(acc)
    history['val_f1'].append(f1)
    history['val_auc'].append(auc)
    history['lr'].append(lr)

    elapsed = time.time() - t0
    print(f'Ep{epoch+1:3d}/{CONFIG["epochs"]} | '
          f'TrL:{train_loss:.4f} VaL:{val_loss:.4f} | '
          f'Acc:{acc*100:.1f}% F1:{f1*100:.2f}% AUC:{auc:.4f} | '
          f'LR:{lr:.2e} | {elapsed:.1f}s')

    if f1 > best_f1:
        best_f1 = f1; patience_counter = 0
        raw = model._orig_mod if hasattr(model, '_orig_mod') else model
        best_state = copy.deepcopy(raw.state_dict())
        torch.save(best_state, 'cawt_v2_best.pth')
        print(f'  >>> NEW BEST F1={best_f1*100:.2f}% saved!')
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['patience']:
            print(f'  Early stopping at epoch {epoch+1}')
            break

print(f'\nTraining complete! Best Val F1: {best_f1*100:.2f}%')

In [ ]:
# Cell 11: Load best model + TTA Evaluation
raw_model = CAWT(CONFIG).to(DEVICE)
raw_model.load_state_dict(best_state)
raw_model.eval()

tta_aug = Augmentor(p=0.6)

def predict_with_tta(model, X, tta_n=10, bs=256):
    probs = np.zeros((len(X), 5), dtype=np.float64)
    # Clean pass
    with torch.no_grad():
        for s in range(0, len(X), bs):
            b = torch.tensor(X[s:s+bs], dtype=torch.float32).unsqueeze(1).to(DEVICE)
            with autocast('cuda'):
                lo = model(b)
            probs[s:s+bs] += lo.float().softmax(1).cpu().numpy()
    # TTA passes
    for t_pass in range(tta_n):
        with torch.no_grad():
            for s in range(0, len(X), bs):
                b = torch.tensor(X[s:s+bs], dtype=torch.float32).unsqueeze(1).to(DEVICE)
                b = tta_aug(b)
                with autocast('cuda'):
                    lo = model(b)
                probs[s:s+bs] += lo.float().softmax(1).cpu().numpy()
    probs /= (tta_n + 1)
    return probs

print('Running TTA x10 on validation set...')
y_probs = predict_with_tta(raw_model, X_val, tta_n=CONFIG['tta_n'])
y_pred = y_probs.argmax(1)

acc, f1, auc, sens, spec = calc_metrics(y_val, y_pred, y_probs)
print(f'\nFINAL RESULTS (Best Model + TTA x10)')
print('=' * 55)
print(f'Accuracy : {acc*100:.2f}%')
print(f'Macro F1 : {f1*100:.2f}%')
print(f'AUROC    : {auc:.4f}')
for i, n in enumerate(CLASS_NAMES):
    print(f'  {n:25s} Sens:{sens[i]*100:.1f}% Spec:{spec[i]*100:.1f}%')
print()
print(classification_report(y_val, y_pred, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
# Cell 12: Training History Plots
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].plot(history['train_loss'], label='Train Loss', color='#2196F3')
axes[0, 0].plot(history['val_loss'], label='Val Loss', color='#F44336')
axes[0, 0].set_title('Loss'); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history['val_acc'], label='Val Accuracy', color='#4CAF50')
axes[0, 1].set_title('Val Accuracy'); axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(history['val_f1'], label='Val Macro F1', color='#FF9800')
axes[1, 0].axhline(0.90, color='red', linestyle='--', alpha=0.5, label='F1=90%')
axes[1, 0].set_title('Val Macro F1'); axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(history['lr'], label='Learning Rate', color='#9C27B0')
axes[1, 1].set_title('Learning Rate'); axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(f'CAWT v2 Training | Best F1={best_f1*100:.2f}%', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 13: Confusion Matrix
cm = confusion_matrix(y_val, y_pred)
cmn = cm.astype('float') / cm.sum(1, keepdims=True)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
sns.heatmap(cmn, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[0].set_title('Counts'); axes[1].set_title('Normalized')
plt.suptitle(f'CAWT v2 | F1={f1*100:.2f}% | AUROC={auc:.4f}', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 14: Export to TorchScript for Android
raw_model.eval().cpu()
with torch.no_grad():
    dummy = torch.randn(1, 1, 187)
    traced = torch.jit.trace(raw_model, dummy)
    out = traced(dummy)
    assert out.shape == (1, 5)
    print(f'Trace OK | output: {list(out.shape)}')

traced.save('cawt_v2_mobile.pt')
torch.save(best_state, 'cawt_v2_best.pth')
fsize = os.path.getsize('cawt_v2_mobile.pt') / 1e6
print(f'\nSaved: cawt_v2_mobile.pt ({fsize:.1f} MB)')
print('Saved: cawt_v2_best.pth (weights only)')
print('\nCopy cawt_v2_mobile.pt to: ECGClassifier/app/src/main/assets/cawt_mobile.pt')